# Step 1 — Preprocessing

Reads the raw dataset from `../../data/raw_data/diabetes_dataset.csv`, applies quality filters, and engineers BMI and age bucket features. Outputs a 6-column `final_df` Parquet to `../../data/final_df`.

> Neither the raw CSV nor the output Parquet are committed to the repository. The CSV must be downloaded to `data/raw_data/` before running this notebook.

**Pipeline position:** this output feeds directly into `feature_engineering.py`. On GCP, upload the output to `gs://team10-diabetes-data/processed/final_df` before submitting the distributed jobs.

### 1. Spark Session

In [3]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("DiabetesPreprocessing") \
    .config("spark.executor.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", True) \
    .getOrCreate()


### 2. Paths

In [9]:
# Resolve paths relative to this notebook's location
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))

INPUT_CSV       = os.path.join(NOTEBOOK_DIR, "..", "..", "data", "raw_data", "diabetes_dataset.csv")
OUTPUT_FINAL_DF = os.path.join(NOTEBOOK_DIR, "..", "..", "data", "final_df")

print("Input :", os.path.normpath(INPUT_CSV))
print("Output:", os.path.normpath(OUTPUT_FINAL_DF))

Input : /Users/swornimbasnet/Desktop/sjsu/sp2026/cs131/Team10-ComprehensiveDiabetesClinicalDataset-health/data/raw_data/diabetes_dataset.csv
Output: /Users/swornimbasnet/Desktop/sjsu/sp2026/cs131/Team10-ComprehensiveDiabetesClinicalDataset-health/data/final_df


### 3. Data Ingestion

In [10]:
df_raw = spark.read.csv(INPUT_CSV, header=True, inferSchema=True)

df_raw.printSchema()
df_raw.show(5)
print("Row count:", df_raw.count())

root
 |-- year: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- age: double (nullable = true)
 |-- location: string (nullable = true)
 |-- race:AfricanAmerican: integer (nullable = true)
 |-- race:Asian: integer (nullable = true)
 |-- race:Caucasian: integer (nullable = true)
 |-- race:Hispanic: integer (nullable = true)
 |-- race:Other: integer (nullable = true)
 |-- hypertension: integer (nullable = true)
 |-- heart_disease: integer (nullable = true)
 |-- smoking_history: string (nullable = true)
 |-- bmi: double (nullable = true)
 |-- hbA1c_level: double (nullable = true)
 |-- blood_glucose_level: integer (nullable = true)
 |-- diabetes: integer (nullable = true)

+----+------+----+--------+--------------------+----------+--------------+-------------+----------+------------+-------------+---------------+-----+-----------+-------------------+--------+
|year|gender| age|location|race:AfricanAmerican|race:Asian|race:Caucasian|race:Hispanic|race:Other|hypertension|h

### 4. Data Cleaning & Quality Filter

In [11]:
critical_cols = ["diabetes", "bmi", "age", "hypertension", "heart_disease"]
df_clean = df_raw.na.drop(subset=critical_cols)

df_filtered = df_clean.filter(
    (F.col("diabetes").isin(0, 1)) &
    (F.col("hypertension").isin(0, 1)) &
    (F.col("heart_disease").isin(0, 1)) &
    (F.col("bmi").between(10, 80))
)

before = df_raw.count()
after  = df_filtered.count()
print(f"Rows before: {before}")
print(f"Rows after:  {after}")
print(f"Dropped:     {before - after}")

Rows before: 100000
Rows after:  99991
Dropped:     9


### 5. BMI & Age Buckets

In [12]:
df_with_features = (
    df_filtered
    .withColumn(
        "bmi_category",
        F.when(F.col("bmi") < 18.5,               "underweight")
         .when(F.col("bmi").between(18.5, 24.9),  "healthy")
         .when(F.col("bmi").between(25.0, 29.9),  "overweight")
         .otherwise("obese")
    )
    .withColumn(
        "age_bucket",
        F.when(F.col("age") < 18,                 "children")
         .when(F.col("age").between(18, 30),       "youth")
         .when(F.col("age").between(31, 60),       "adult")
         .otherwise("senior")
    )
)

df_with_features.select("age", "age_bucket", "bmi", "bmi_category", "diabetes").show(10)

+----+----------+-----+------------+--------+
| age|age_bucket|  bmi|bmi_category|diabetes|
+----+----------+-----+------------+--------+
|32.0|     adult|27.32|  overweight|       0|
|29.0|     youth|19.95|     healthy|       0|
|18.0|     youth|23.76|     healthy|       0|
|41.0|     adult|27.32|  overweight|       0|
|52.0|     adult|23.75|     healthy|       0|
|66.0|    senior|27.32|  overweight|       0|
|49.0|     adult|24.34|     healthy|       0|
|15.0|  children|20.98|     healthy|       0|
|51.0|     adult|38.14|       obese|       0|
|42.0|     adult|27.32|  overweight|       0|
+----+----------+-----+------------+--------+
only showing top 10 rows


### 6. Validation — Frequency Tables

In [15]:
diabetic = df_with_features.filter(F.col("diabetes") == 1)

print("BMI Category (diabetic patients):")
diabetic.groupBy("bmi_category") \
    .agg(F.count("*").alias("frequency")) \
    .orderBy("bmi_category") \
    .show()

print("Age Bucket (diabetic patients):")
diabetic.groupBy("age_bucket") \
    .agg(F.count("*").alias("frequency")) \
    .orderBy("age_bucket") \
    .show()

print("Hypertension (diabetic patients):")
diabetic.groupBy("hypertension") \
    .agg(F.count("*").alias("frequency")) \
    .orderBy("hypertension") \
    .show()

print("Heart Disease (diabetic patients):")
diabetic.groupBy("heart_disease") \
    .agg(F.count("*").alias("frequency")) \
    .orderBy("heart_disease") \
    .show()

BMI Category (diabetic patients):
+------------+---------+
|bmi_category|frequency|
+------------+---------+
|     healthy|      839|
|       obese|     4299|
|  overweight|     3295|
| underweight|       64|
+------------+---------+

Age Bucket (diabetic patients):
+----------+---------+
|age_bucket|frequency|
+----------+---------+
|     adult|     3477|
|  children|       82|
|    senior|     4719|
|     youth|      219|
+----------+---------+

Hypertension (diabetic patients):
+------------+---------+
|hypertension|frequency|
+------------+---------+
|           0|     6409|
|           1|     2088|
+------------+---------+

Heart Disease (diabetic patients):
+-------------+---------+
|heart_disease|frequency|
+-------------+---------+
|            0|     7230|
|            1|     1267|
+-------------+---------+



### 7. Build & Save `final_df`

Narrow to the 6 columns consumed by `feature_engineering.py`.

In [14]:
SKINNY_COLS = ["age_bucket", "bmi", "bmi_category", "hypertension", "heart_disease", "diabetes"]

final_df = df_with_features.select(*SKINNY_COLS)
final_df.show(10, truncate=False)

final_df.write.mode("overwrite").parquet(OUTPUT_FINAL_DF)
print(f"Saved {final_df.count()} rows → {os.path.normpath(OUTPUT_FINAL_DF)}")

+----------+-----+------------+------------+-------------+--------+
|age_bucket|bmi  |bmi_category|hypertension|heart_disease|diabetes|
+----------+-----+------------+------------+-------------+--------+
|adult     |27.32|overweight  |0           |0            |0       |
|youth     |19.95|healthy     |0           |0            |0       |
|youth     |23.76|healthy     |0           |0            |0       |
|adult     |27.32|overweight  |0           |0            |0       |
|adult     |23.75|healthy     |0           |0            |0       |
|senior    |27.32|overweight  |0           |0            |0       |
|adult     |24.34|healthy     |0           |0            |0       |
|children  |20.98|healthy     |0           |0            |0       |
|adult     |38.14|obese       |0           |0            |0       |
|adult     |27.32|overweight  |0           |0            |0       |
+----------+-----+------------+------------+-------------+--------+
only showing top 10 rows


Saved 99991 rows → /Users/swornimbasnet/Desktop/sjsu/sp2026/cs131/Team10-ComprehensiveDiabetesClinicalDataset-health/data/final_df


### 8. Stop Spark

In [ ]:
spark.stop()
print("Done. Next step: feature_engineering.py")